# Inventario de items del Workspace -> config.DOMAIN_OBJECTS

Flujo:
1. Leer los items del workspace con `sempy` (`fabric.list_items`).
2. Resolver `SOURCE_TYPE_ID` contra `config.SOURCE_TYPES`.
3. Escribir la foto del día en **`config.STG_DOMAIN_OBJECTS`** (staging, se pisa en cada corrida).
4. Ejecutar **`config.usp_SYNC_DOMAIN_OBJECTS`**: altas, cambios y bajas lógicas (`ACTIVE_FLAG = 0`).

La tabla definitiva **nunca** se escribe con `overwrite` desde Spark: así conserva `IDENTITY`, PK e índices.
Requisito: haber corrido antes `01_ddl_domain_objects.sql`.

## Parámetros

In [ ]:
# Celda de parámetros (marcala como "parameter cell" para pasarla desde un pipeline)
WORKSPACES = ["<nombre-del-workspace>"]   # se pueden inventariar varios
MIN_ROWS_EXPECTED = 1                  # guardia: si vienen menos filas, no se sincroniza

SQL_SERVER   = "<servidor>.database.fabric.microsoft.com"   # SQL database > Settings > Connection strings
SQL_DATABASE = "<nombre-de-la-base>"

STG_TABLE    = "config.STG_DOMAIN_OBJECTS"
SYNC_PROC    = "config.usp_SYNC_DOMAIN_OBJECTS"

## 1. Items del workspace

In [ ]:
import pandas as pd
from sempy import fabric

frames = []
for ws in WORKSPACES:
    df_ws = fabric.list_items(workspace=ws)
    print(f"{ws}: {len(df_ws)} items")
    frames.append(df_ws)

df_pandas = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# Si no vino nada cortamos acá: mejor fallar que marcar todo como borrado
if len(df_pandas) < MIN_ROWS_EXPECTED:
    raise ValueError(f"list_items devolvió {len(df_pandas)} filas. Revisá permisos o el nombre del workspace.")

df_spark = spark.createDataFrame(df_pandas)
df_spark.printSchema()

In [ ]:
from pyspark.sql import functions as F

df_items = (
    df_spark.select(
        F.col("Workspace Id").cast("string").alias("WORKSPACE_ID"),
        F.col("Display Name").cast("string").alias("OBJECT_NAME"),
        F.col("Id").cast("string").alias("FABRIC_ITEM_GUID"),
        F.col("Type").cast("string").alias("ARTIFACT_TYPE"),
    )
    .dropDuplicates(["FABRIC_ITEM_GUID"])
)
# Nota: Spark resuelve nombres de columna sin distinguir mayúsculas por defecto,
# por eso "Workspace Id" y "WORKSPACE ID" funcionan igual.

## 2. Conexión a SQL Database y tipos de origen

In [ ]:
import com.microsoft.sqlserver.jdbc.spark  # registra el conector de Fabric

# trustServerCertificate=false: validamos el certificado del servidor
url = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;"
    f"database={SQL_DATABASE};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.fabric.microsoft.com;"
    "loginTimeout=30;"
)

def read_sql(table):
    return (spark.read
            .format("com.microsoft.sqlserver.jdbc.spark")
            .option("url", url)
            .option("dbtable", table)
            .load())

df_source_types = read_sql("config.SOURCE_TYPES").select("SOURCE_TYPE_ID", "SOURCE_TYPE_NAME")

In [ ]:
df_stage = (
    df_items.alias("d")
    .join(F.broadcast(df_source_types.alias("s")),
          F.col("d.ARTIFACT_TYPE") == F.col("s.SOURCE_TYPE_NAME"),
          "left")
    .select(
        "d.WORKSPACE_ID", "d.OBJECT_NAME", "d.FABRIC_ITEM_GUID",
        F.col("s.SOURCE_TYPE_ID").cast("int").alias("SOURCE_TYPE_ID"),
        "d.ARTIFACT_TYPE",
        F.current_timestamp().alias("EXTRACTED_AT"),
    )
)

# Aviso de tipos de Fabric que faltan en config.SOURCE_TYPES
sin_tipo = df_stage.filter(F.col("SOURCE_TYPE_ID").isNull()).groupBy("ARTIFACT_TYPE").count()
if sin_tipo.count() > 0:
    print("Tipos sin SOURCE_TYPE_ID (agregalos a config.SOURCE_TYPES):")
    display(sin_tipo)

print("Filas a staging:", df_stage.count())

## 3. Escribir staging

La staging es descartable: acá sí se puede usar `overwrite`.

In [ ]:
(
    df_stage.write
    .format("com.microsoft.sqlserver.jdbc.spark")
    .mode("overwrite")
    .option("url", url)
    .option("dbtable", STG_TABLE)
    .save()
)

## 4. Sincronizar: altas, cambios y bajas lógicas

Se ejecuta el stored procedure por JDBC con un token de Entra ID de la identidad que corre el notebook.
Alternativa sin código: en un Data Pipeline, actividad **Notebook** (pasos 1 a 3) seguida de una actividad **Stored procedure**.

In [ ]:
# [verificar] audiencia del token para SQL database en Fabric; si falla, probá "https://database.windows.net/.default"
token = notebookutils.credentials.getToken("https://database.windows.net/")

jvm = spark._jvm
props = jvm.java.util.Properties()
props.setProperty("accessToken", token)

conn = jvm.java.sql.DriverManager.getConnection(url, props)
try:
    stmt = conn.prepareCall(f"{{call {SYNC_PROC}(?)}}")
    stmt.setInt(1, MIN_ROWS_EXPECTED)
    has_rs = stmt.execute()
    if has_rs:
        rs = stmt.getResultSet()
        while rs.next():
            print(f"Staging: {rs.getInt('FILAS_STAGING')} | "
                  f"Altas: {rs.getInt('ALTAS')} | "
                  f"Cambios/bajas: {rs.getInt('CAMBIOS_O_BAJAS')}")
finally:
    conn.close()

## 5. Validación

In [ ]:
df_target = read_sql("config.DOMAIN_OBJECTS")

resumen = df_target.groupBy("WORKSPACE_ID", "ACTIVE_FLAG").count().orderBy("WORKSPACE_ID", "ACTIVE_FLAG")
display(resumen)

# Chequeo: los activos de la tabla (de los workspaces inventariados) deben coincidir con el workspace
guids_ws = df_stage.select("FABRIC_ITEM_GUID")
activos  = (df_target.filter("ACTIVE_FLAG = true")
                     .join(df_stage.select("WORKSPACE_ID").distinct(), "WORKSPACE_ID")
                     .select("FABRIC_ITEM_GUID"))
faltan = guids_ws.subtract(activos).count()
sobran = activos.subtract(guids_ws).count()
print(f"Faltan en tabla: {faltan} | Activos que ya no existen: {sobran}")
assert faltan == 0 and sobran == 0, "La tabla no quedó alineada con el workspace"